# AV2 - Etapa 1: ingestão e tokenização
**Grupo 2 - Responsável: Diego Bispo**

Este notebook é independente das células da AV1. No Colab, use **Ambiente de
execução > Executar tudo**. Os artefatos da AV1 são apenas lidos.

- Tokenizer: tiktoken 0.14.0, `cl100k_base`, `encode_ordinary`.
- Entrada: corpus e candidatos completos da AV1 (sem corte Top-k).
- Saída: instâncias de mochila 0/1 com W = 512, 1024, 2048, 4096.
- Não executa ainda os seletores guloso/PD e não chama uma LLM.

W é o orçamento dos textos dos chunks. A integração reservará espaço adicional
para instruções, consulta, citações e resposta do modelo.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/diegofnf/PAA_UFS_2026_2_Bispo_Diego_Marques_Gabriel_Andrade_Laryssa_Santos_Kaio_Melo_Victor.git'
try:
    import google.colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False
REPO_DIR = Path('/content/PAA_AV2') if NO_COLAB else Path.cwd()
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', 'av2', REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / 'av2/1_scripts/preparar_instancias.py').is_file():
    raise RuntimeError('Use uma copia da branch av2; a main da AV1 nao contem esta etapa.')
print('Python em execucao:', sys.executable)
print('Commit em execucao:', subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO_DIR / 'av2/requirements.txt'), 'pytest==9.1.1', 'nltk==3.10.3'], check=True)


In [ ]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/preparar_instancias.py')],
               cwd=REPO_DIR, check=True)
relatorio = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_preparacao.json').read_text(encoding='utf-8'))
print('Consulta:', relatorio['metadados']['consulta'])
print('Tokenizer:', relatorio['metadados']['tokenizer'])
print('Chunks:', relatorio['total_chunks'], '| Candidatos positivos:', relatorio['total_candidatos_positivos'])
for instancia in relatorio['instancias']:
    print(instancia)
print('Tempos de preparacao (s):', relatorio['tempos_segundos'])


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'av2/1_scripts/test_preparar_instancias.py',
                'av1/1_scripts/test_etapa_4.py', '-q',
                '--junitxml=av2/6_evidencias/testes.xml'], cwd=REPO_DIR, check=True)
instancias = json.loads((REPO_DIR / 'av2/3_instancias/instancias.json').read_text(encoding='utf-8'))
assert [i['W'] for i in instancias['instancias']] == [512, 1024, 2048, 4096]
assert all(c['w_i'] > 0 and c['v_i'] > 0 for c in instancias['itens'])
print('Instancias prontas para as etapas de selecao. Veja av2/README.md para o contrato.')


## AV2 - Etapa 2: Seleção de Contexto por Baseline Guloso (Mochila 0/1)
**Grupo 2 - Responsável: Kaio Santos**

Esta etapa consome as instâncias preparadas em `av2/3_instancias/instancias.json` e aplica
a heurística gulosa de densidade de relevância por token ($r_i = v_i / w_i$).

- **Objetivo:** Maximizar a relevância total $\sum v_i x_i$ sujeita a $\sum w_i x_i \le W$ com $x_i \in \{0, 1\}$.
- **Desempate determinístico:** maior densidade $r_i$ (desc), maior $v_i$ (desc), menor $w_i$ (asc), menor `id_chunk` (asc).
- **Varredura contínua:** Se um item não couber no saldo, o algoritmo o pula mas continua avaliando os itens subsequentes menores.
- **Saídas:** `av2/4_selecao_contexto/selecao_gulosa.json` e `av2/5_resultados/relatorio_guloso.json`.


In [ ]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/selecionar_guloso.py')],
               cwd=REPO_DIR, check=True)
relatorio_guloso = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_guloso.json').read_text(encoding='utf-8'))
print('Consulta:', relatorio_guloso['metadados']['consulta'])
print('Total de candidatos avaliados:', relatorio_guloso['metadados']['total_candidatos_positivos'])
print('\nResultados da Seleção Gulosa:')
for cenario in relatorio_guloso['resumo_cenarios']:
    print(f"  [W={cenario['W']:4d}] Chunks: {cenario['total_chunks_selecionados']:2d} | "
          f"Tokens: {cenario['tokens_usados']:4d}/{cenario['W']:4d} (Saldo: {cenario['saldo_W']:3d}, Ocupação: {cenario['taxa_ocupacao_percentual']:5.1f}%) | "
          f"Relevância: {cenario['relevancia_total']:7.4f} | IDs: {cenario['ids_selecionados']}")
print('\nTempos de execução da seleção (s):', relatorio_guloso['tempos_segundos'])


In [ ]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/test_selecionar_guloso.py')],
               cwd=REPO_DIR, check=True)
selecao_gulosa = json.loads((REPO_DIR / 'av2/4_selecao_contexto/selecao_gulosa.json').read_text(encoding='utf-8'))
assert all(c['tokens_usados'] <= c['W'] for c in selecao_gulosa['cenarios'].values())
print('Todos os testes do seletor guloso passaram com sucesso!')


## AV2 - Etapa 3: Seleção Ótima de Contexto por Programação Dinâmica (Mochila 0/1)
**Grupo 2 - Responsáveis: Diego Bispo e Kaio Santos**

Esta etapa resolve o problema da Mochila 0/1 exata via Programação Dinâmica bottom-up,
garantindo a seleção ótima global que maximiza a relevância total sem exceder $W$.

- **Equação de Bellman:** $DP[i][c] = \max(DP[i-1][c],\; DP[i-1][c - w_i] + v_i)$
- **Recuperação de Chunks:** Backtracking determinístico a partir de $DP[n][W]$.
- **Complexidade no Modelo RAM:** Tempo $\mathcal{O}(n \cdot W)$ e Espaço $\mathcal{O}(n \cdot W)$.
- **Saídas:** `av2/4_selecao_contexto/selecao_pd.json` e `av2/5_resultados/relatorio_pd.json`.


In [ ]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/selecionar_pd.py')],
               cwd=REPO_DIR, check=True)
relatorio_pd = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_pd.json').read_text(encoding='utf-8'))
relatorio_g = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_guloso.json').read_text(encoding='utf-8'))
map_g = {c['W']: c for c in relatorio_g['resumo_cenarios']}

print('=== Confronto: Baseline Guloso vs. Programação Dinâmica ===')
for c_pd in relatorio_pd['resumo_cenarios']:
    W = c_pd['W']
    cg = map_g[W]
    ganho_rel = c_pd['relevancia_total'] - cg['relevancia_total']
    ganho_pct = (ganho_rel / cg['relevancia_total'] * 100) if cg['relevancia_total'] > 0 else 0
    print(f"\n[Capacidade W = {W}]")
    print(f"  Guloso: Chunks={cg['total_chunks_selecionados']:2d} | Tokens={cg['tokens_usados']:4d}/{W} (Saldo: {cg['saldo_W']:3d}) | Relevância={cg['relevancia_total']:7.4f}")
    print(f"  PD:     Chunks={c_pd['total_chunks_selecionados']:2d} | Tokens={c_pd['tokens_usados']:4d}/{W} (Saldo: {c_pd['saldo_W']:3d}) | Relevância={c_pd['relevancia_total']:7.4f}")
    print(f"  Ganho da PD: {ganho_rel:+.4f} ({ganho_pct:+.2f}%)")
print('\nTempos de execução da PD (s):', relatorio_pd['tempos_segundos'])


In [ ]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/test_selecionar_pd.py')],
               cwd=REPO_DIR, check=True)
selecao_pd = json.loads((REPO_DIR / 'av2/4_selecao_contexto/selecao_pd.json').read_text(encoding='utf-8'))
assert all(c['tokens_usados'] <= c['W'] for c in selecao_pd['cenarios'].values())
print('Todos os testes da Programação Dinâmica passaram com sucesso!')


## Próximas etapas
- **Etapa 4 — Benchmark Comparativo:** Avaliação sistemática de tempo de execução, consumo de memória RAM, escalabilidade e qualidade de contexto recuperado.
